<a href="https://colab.research.google.com/github/AzadMahmud/DIP-Lab/blob/main/assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [2]:
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "code":
        ROOT = ROOT.parent

OUT = ROOT / "outputs"
FIG = OUT / "figures"
MET = OUT / "metrics"
MOD = OUT / "models"
for d in [FIG, MET, MOD]:
    d.mkdir(parents=True, exist_ok=True)

In [4]:
def load_cifar10(n_train=20000, n_val=2000, n_test=2000):
    from datasets import load_dataset
    import numpy as np

    ds = load_dataset("cifar10")

    xtr = np.array(ds["train"]["img"]).astype("float32") / 255.0
    xte = np.array(ds["test"]["img"]).astype("float32") / 255.0

    xval = xtr[-n_val:]
    xtr = xtr[:n_train]
    xte = xte[:n_test]

    return xtr, xval, xte

In [5]:
def add_gaussian(x, sigma):
    y = x + np.random.normal(0.0, sigma, size=x.shape).astype(np.float32)
    return np.clip(y, 0.0, 1.0)

In [6]:
def add_saltpepper(x, amount):
    y = x.copy()
    n = int(amount * x.shape[0] * x.shape[1] * x.shape[2] * x.shape[3])
    idx = np.unravel_index(np.random.randint(0, x.size, size=n), x.shape)
    y[idx] = np.random.randint(0, 2, size=n).astype(np.float32)
    return y


In [7]:
def make_dae(upsampling: str):
    inp = layers.Input((32, 32, 3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    z = layers.Conv2D(128, 3, padding="same", activation="relu")(x)

    if upsampling == "transpose":
        x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(z)
        x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)
    else:  # upsample
        x = layers.UpSampling2D()(z)
        x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
        x = layers.UpSampling2D()(x)
        x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)

    out = layers.Conv2D(3, 3, padding="same", activation="sigmoid")(x)
    m = models.Model(inp, out, name=f"dae_{upsampling}")
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return m

In [8]:
def psnr(x, y):
    return float(tf.image.psnr(x, y, max_val=1.0).numpy().mean())

In [9]:
def run_one(noise: str, level: float, upsampling: str, epochs=8, bs=128):
    xtr, xval, xte = load_cifar10()
    if noise == "gaussian":
        xtr_n, xval_n, xte_n = add_gaussian(xtr, level), add_gaussian(xval, level), add_gaussian(xte, level)
    else:
        xtr_n, xval_n, xte_n = add_saltpepper(xtr, level), add_saltpepper(xval, level), add_saltpepper(xte, level)

    m = make_dae(upsampling)
    cbs = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    t0 = time.time()
    h = m.fit(xtr_n, xtr, validation_data=(xval_n, xval), epochs=epochs, batch_size=bs, verbose=1, callbacks=cbs)
    t1 = time.time()
    te_loss = float(m.evaluate(xte_n, xte, verbose=0))

    den = m.predict(xte_n[:32], verbose=0)
    p = psnr(xte[:32], den)

    tag = f"{noise}_{level}_{upsampling}"
    pd.DataFrame(h.history).to_csv(MET / f"history_{tag}.csv", index=False)
    m.save(MOD / f"{tag}.keras")

    fig, ax = plt.subplots(3, 8, figsize=(12, 5))
    for i in range(8):
        ax[0, i].imshow(xte[i])
        ax[1, i].imshow(xte_n[i])
        ax[2, i].imshow(den[i])
        for r in range(3):
            ax[r, i].axis("off")
    ax[0, 0].set_ylabel("clean", fontsize=10)
    ax[1, 0].set_ylabel("noisy", fontsize=10)
    ax[2, 0].set_ylabel("denoise", fontsize=10)
    fig.suptitle(f"{tag} | test_mse={te_loss:.4f} | psnr={p:.2f}")
    fig.tight_layout()
    fig.savefig(FIG / f"samples_{tag}.png", dpi=220)
    plt.close(fig)

    tf.keras.backend.clear_session()
    return {
        "noise": noise,
        "level": level,
        "upsampling": upsampling,
        "epochs_ran": len(h.history["loss"]),
        "val_loss_best": float(min(h.history["val_loss"])),
        "test_mse": te_loss,
        "psnr": p,
        "train_time_sec": float(t1 - t0),
        "tag": tag,
    }

In [10]:
def main():
    rows = []
    for noise in ["gaussian", "saltpepper"]:
        for level in [0.1, 0.3]:
            for up in ["transpose", "upsample"]:
                print("Running", noise, level, up)
                rows.append(run_one(noise, level, up))

    df = pd.DataFrame(rows).sort_values(["test_mse", "psnr"], ascending=[True, False])
    df.to_csv(MET / "comparison.csv", index=False)
    best = df.iloc[0].to_dict()
    (MET / "summary.json").write_text(
        json.dumps(
            {
                "best_tag": best["tag"],
                "best_test_mse": float(best["test_mse"]),
                "best_psnr": float(best["psnr"]),
                "note": "Higher noise level should worsen reconstruction; compare transpose vs upsample in comparison.csv",
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Done. Outputs saved in", OUT)


if __name__ == "__main__":
    main()

Running gaussian 0.1 transpose


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Epoch 1/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - loss: 0.0209 - val_loss: 0.0097
Epoch 2/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0082 - val_loss: 0.0069
Epoch 3/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0067 - val_loss: 0.0059
Epoch 4/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0057 - val_loss: 0.0058
Epoch 5/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0053 - val_loss: 0.0051
Epoch 6/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0052 - val_loss: 0.0050
Epoch 7/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0049 - val_loss: 0.0049
Epoch 8/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0048 - val_loss: 0.0046
Running gaussian 0.1 upsample
Epoch 1/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - loss: 0.0166 - val_loss: 0.0082
Epoch 2/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.0067 - val_loss: 0.0069
Epoch 3/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.0056 - val_loss: 0.0051
Epoch 4/8
157/157 ━━━━

Running saltpepper 0.1 upsample
Epoch 1/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - loss: 0.0177 - val_loss: 0.0088
Epoch 2/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0081 - val_loss: 0.0072
Epoch 3/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0070 - val_loss: 0.0070
Epoch 4/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0063 - val_loss: 0.0060
Epoch 5/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0059 - val_loss: 0.0056
Epoch 6/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0056 - val_loss: 0.0053
Epoch 7/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0053 - val_loss: 0.0051
Epoch 8/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0051 - val_loss: 0.0049


Running saltpepper 0.3 transpose
Epoch 1/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - loss: 0.0242 - val_loss: 0.0166
Epoch 2/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0129 - val_loss: 0.0111
Epoch 3/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0107 - val_loss: 0.0112
Epoch 4/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0100 - val_loss: 0.0095
Epoch 5/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.0094 - val_loss: 0.0095
Epoch 6/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0090 - val_loss: 0.0090
Epoch 7/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0086 - val_loss: 0.0085
Epoch 8/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0083 - val_loss: 0.0084
Running saltpepper 0.3 upsample
Epoch 1/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - loss: 0.0189 - val_loss: 0.0119
Epoch 2/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0111 - val_loss: 0.0107
Epoch 3/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0098 - val_lo